# Quantized GEMV kernel ladder: T4

**Before running anything: Runtime, Change runtime type, and select the
T4 GPU.** Colab hands out whatever it has free unless told otherwise, and
this study has already been bitten by that: two Kaggle sessions that asked for a
T4 both returned a P100, and an older `out-t4` results directory turned out to
contain P100 data. The next cell checks what you actually got and says so
loudly.

One decode-path operation, `y = W x` with W quantized to ggml `q8_0` or `q4_0`,
written nine ways, identical sources to the RTX 3050 run. Results are written to
a filename derived from the **actual** device, never the expected one, so a
substituted runtime cannot produce a mislabelled file.

Every number printed below was measured in this session on the device this
notebook names. Nothing is extrapolated.

In [ ]:
import json, os, subprocess, sys, pathlib, textwrap, re
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(subprocess.run(['nvcc','--version'], capture_output=True, text=True).stdout)

In [ ]:
# Which GPU did this session actually get? The output filename comes from this,
# not from what the notebook was named, so a substituted runtime is impossible
# to mislabel afterwards.
import torch
EXPECT = 'Tesla T4'
actual = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
slug = re.sub(r'[^a-z0-9]+', '_', actual.lower()).strip('_')
OUT_NAME = f"colab_{slug}_results.json"
print("actual device :", actual, f"sm_{cc[0]}{cc[1]}")
print("output file   :", OUT_NAME)
if EXPECT and EXPECT.lower() not in actual.lower():
    print()
    print("!" * 72)
    print(f"WRONG RUNTIME: this notebook targets {EXPECT} but got {actual}.")
    print("Fix: Runtime > Change runtime type > pick the right GPU > Run all.")
    print("The run below will still work and the file will be named for the")
    print("device you actually got, so nothing gets mislabelled either way.")
    print("!" * 72)
else:
    print("runtime matches the target")

In [ ]:
SRC = pathlib.Path('/tmp/qgemv'); SRC.mkdir(exist_ok=True)
KERNELS = r'''// Quantized GEMV kernels for the batch-1 LLM decode path.
//
// One operation: y = W * x, where W is a quantized [M, K] weight matrix and x is
// an fp32 activation vector of length K. This is the operation that dominates
// batch-1 decode, and it is memory bound on every GPU ever built.
//
// The block layouts are byte-identical to the ggml block_q8_0 and block_q4_0
// structs, so a row of W here is a row of W in a GGUF file. That is deliberate:
// the point of the AoS-vs-SoA comparison below is to measure what the on-disk
// layout costs at the kernel level.

#pragma once
#include <cuda_fp16.h>
#include <cstdint>

#define QK 32  // ggml quantization block size, both formats

// ---------------------------------------------------------------- layouts

// 34 bytes. Note it is not a multiple of 4, let alone 16.
struct block_q8_0 { __half d; int8_t qs[QK]; };
// 18 bytes. Same problem.
struct block_q4_0 { __half d; uint8_t qs[QK / 2]; };

static_assert(sizeof(block_q8_0) == 34, "block_q8_0 must match ggml");
static_assert(sizeof(block_q4_0) == 18, "block_q4_0 must match ggml");

__device__ __forceinline__ float warp_reduce(float v) {
#pragma unroll
  for (int off = 16; off > 0; off >>= 1) v += __shfl_down_sync(0xffffffffu, v, off);
  return v;
}

// ================================================================ Q8_0

// v0: one thread per row, ggml AoS layout, scalar loop.
// Adjacent threads are (K/32)*34 bytes apart, so every load is a separate
// memory transaction. This is the baseline, and it is meant to be bad.
__global__ void qgemv_q8_v0_naive(const block_q8_0* __restrict__ W,
                                  const float* __restrict__ x,
                                  float* __restrict__ y, int M, int K) {
  const int row = blockIdx.x * blockDim.x + threadIdx.x;
  if (row >= M) return;
  const int nb = K / QK;
  const block_q8_0* w = W + (size_t)row * nb;
  float acc = 0.f;
  for (int b = 0; b < nb; ++b) {
    float s = 0.f;
    for (int j = 0; j < QK; ++j) s += (float)w[b].qs[j] * x[b * QK + j];
    acc += __half2float(w[b].d) * s;
  }
  y[row] = acc;
}

// v1: one warp per row, ggml AoS layout. Lane l takes blocks l, l+32, l+64...
// so in each iteration the warp reads 32*34 = 1088 contiguous bytes. The
// accesses coalesce in aggregate even though no single lane load is wide.
__global__ void qgemv_q8_v1_warp(const block_q8_0* __restrict__ W,
                                 const float* __restrict__ x,
                                 float* __restrict__ y, int M, int K) {
  const int warps = blockDim.x / 32;
  const int lane = threadIdx.x & 31;
  const int row = blockIdx.x * warps + (threadIdx.x >> 5);
  if (row >= M) return;
  const int nb = K / QK;
  const block_q8_0* w = W + (size_t)row * nb;
  float acc = 0.f;
  for (int b = lane; b < nb; b += 32) {
    const block_q8_0 blk = w[b];
    float s = 0.f;
#pragma unroll
    for (int j = 0; j < QK; ++j) s += (float)blk.qs[j] * x[b * QK + j];
    acc += __half2float(blk.d) * s;
  }
  acc = warp_reduce(acc);
  if (lane == 0) y[row] = acc;
}

// v2: as v1, but the activation vector is staged in shared memory once per
// block and reused by every row the block owns. Weight traffic is unchanged;
// this only tests whether x traffic was costing anything.
__global__ void qgemv_q8_v2_smem_x(const block_q8_0* __restrict__ W,
                                   const float* __restrict__ x,
                                   float* __restrict__ y, int M, int K) {
  extern __shared__ float sx[];
  for (int i = threadIdx.x; i < K; i += blockDim.x) sx[i] = x[i];
  __syncthreads();
  const int warps = blockDim.x / 32;
  const int lane = threadIdx.x & 31;
  const int row = blockIdx.x * warps + (threadIdx.x >> 5);
  if (row >= M) return;
  const int nb = K / QK;
  const block_q8_0* w = W + (size_t)row * nb;
  float acc = 0.f;
  for (int b = lane; b < nb; b += 32) {
    const block_q8_0 blk = w[b];
    float s = 0.f;
#pragma unroll
    for (int j = 0; j < QK; ++j) s += (float)blk.qs[j] * sx[b * QK + j];
    acc += __half2float(blk.d) * s;
  }
  acc = warp_reduce(acc);
  if (lane == 0) y[row] = acc;
}

// v3: SoA. Quants live in one contiguous int8 plane, scales in another.
// Each lane now issues a 16-byte load, so the warp reads 512 aligned
// contiguous bytes per instruction. This is a layout change, not a scheduling
// change: v1 and v3 do the same arithmetic in the same order.
__global__ void qgemv_q8_v3_soa_vec(const int8_t* __restrict__ Q,
                                    const __half* __restrict__ D,
                                    const float* __restrict__ x,
                                    float* __restrict__ y, int M, int K) {
  const int warps = blockDim.x / 32;
  const int lane = threadIdx.x & 31;
  const int row = blockIdx.x * warps + (threadIdx.x >> 5);
  if (row >= M) return;
  const int nb = K / QK;
  const int4* q4 = reinterpret_cast<const int4*>(Q + (size_t)row * K);
  const __half* d = D + (size_t)row * nb;
  const int nvec = K / 16;  // 16 int8 per lane load
  float acc = 0.f;
  for (int v = lane; v < nvec; v += 32) {
    int4 raw = q4[v];
    const int8_t* q = reinterpret_cast<const int8_t*>(&raw);
    const int base = v * 16;
    float s = 0.f;
#pragma unroll
    for (int j = 0; j < 16; ++j) s += (float)q[j] * x[base + j];
    acc += __half2float(d[base / QK]) * s;
  }
  acc = warp_reduce(acc);
  if (lane == 0) y[row] = acc;
}

// v4: SoA with WPR warps cooperating on one row, so that a small M still fills
// the machine. Partial sums are reduced through shared memory.
template <int WPR>
__global__ void qgemv_q8_v4_split(const int8_t* __restrict__ Q,
                                  const __half* __restrict__ D,
                                  const float* __restrict__ x,
                                  float* __restrict__ y, int M, int K) {
  extern __shared__ float part[];
  const int lane = threadIdx.x & 31;
  const int wid = threadIdx.x >> 5;  // warp id inside the block
  const int rows_per_block = blockDim.x / (32 * WPR);
  const int row_local = wid / WPR;
  const int sub = wid % WPR;  // which warp of this row
  const int row = blockIdx.x * rows_per_block + row_local;
  const int nb = K / QK;
  const int nvec = K / 16;
  float acc = 0.f;
  if (row < M) {
    const int4* q4 = reinterpret_cast<const int4*>(Q + (size_t)row * K);
    const __half* d = D + (size_t)row * nb;
    for (int v = lane + sub * 32; v < nvec; v += 32 * WPR) {
      int4 raw = q4[v];
      const int8_t* q = reinterpret_cast<const int8_t*>(&raw);
      const int base = v * 16;
      float s = 0.f;
#pragma unroll
      for (int j = 0; j < 16; ++j) s += (float)q[j] * x[base + j];
      acc += __half2float(d[base / QK]) * s;
    }
  }
  acc = warp_reduce(acc);
  if (lane == 0) part[wid] = acc;
  __syncthreads();
  if (row < M && sub == 0 && lane == 0) {
    float t = 0.f;
#pragma unroll
    for (int i = 0; i < WPR; ++i) t += part[row_local * WPR + i];
    y[row] = t;
  }
}

// ================================================================ Q4_0

// v0: one thread per row, ggml AoS layout. Same shape as the q8 baseline.
__global__ void qgemv_q4_v0_naive(const block_q4_0* __restrict__ W,
                                  const float* __restrict__ x,
                                  float* __restrict__ y, int M, int K) {
  const int row = blockIdx.x * blockDim.x + threadIdx.x;
  if (row >= M) return;
  const int nb = K / QK;
  const block_q4_0* w = W + (size_t)row * nb;
  float acc = 0.f;
  for (int b = 0; b < nb; ++b) {
    float s = 0.f;
    for (int j = 0; j < QK / 2; ++j) {
      const uint8_t p = w[b].qs[j];
      s += (float)((int)(p & 0x0F) - 8) * x[b * QK + j];
      s += (float)((int)(p >> 4) - 8) * x[b * QK + j + 16];
    }
    acc += __half2float(w[b].d) * s;
  }
  y[row] = acc;
}

// v1: one warp per row, ggml AoS layout.
__global__ void qgemv_q4_v1_warp(const block_q4_0* __restrict__ W,
                                 const float* __restrict__ x,
                                 float* __restrict__ y, int M, int K) {
  const int warps = blockDim.x / 32;
  const int lane = threadIdx.x & 31;
  const int row = blockIdx.x * warps + (threadIdx.x >> 5);
  if (row >= M) return;
  const int nb = K / QK;
  const block_q4_0* w = W + (size_t)row * nb;
  float acc = 0.f;
  for (int b = lane; b < nb; b += 32) {
    const block_q4_0 blk = w[b];
    float s = 0.f;
#pragma unroll
    for (int j = 0; j < QK / 2; ++j) {
      const uint8_t p = blk.qs[j];
      s += (float)((int)(p & 0x0F) - 8) * x[b * QK + j];
      s += (float)((int)(p >> 4) - 8) * x[b * QK + j + 16];
    }
    acc += __half2float(blk.d) * s;
  }
  acc = warp_reduce(acc);
  if (lane == 0) y[row] = acc;
}

// v3: SoA. One 16-byte lane load is exactly one q4_0 block of quants, which is
// the tidiest the two formats get.
__global__ void qgemv_q4_v3_soa_vec(const uint8_t* __restrict__ Q,
                                    const __half* __restrict__ D,
                                    const float* __restrict__ x,
                                    float* __restrict__ y, int M, int K) {
  const int warps = blockDim.x / 32;
  const int lane = threadIdx.x & 31;
  const int row = blockIdx.x * warps + (threadIdx.x >> 5);
  if (row >= M) return;
  const int nb = K / QK;
  const int4* q4 = reinterpret_cast<const int4*>(Q + (size_t)row * (K / 2));
  const __half* d = D + (size_t)row * nb;
  float acc = 0.f;
  for (int b = lane; b < nb; b += 32) {
    int4 raw = q4[b];
    const uint8_t* p = reinterpret_cast<const uint8_t*>(&raw);
    const int base = b * QK;
    float s = 0.f;
#pragma unroll
    for (int j = 0; j < QK / 2; ++j) {
      s += (float)((int)(p[j] & 0x0F) - 8) * x[base + j];
      s += (float)((int)(p[j] >> 4) - 8) * x[base + j + 16];
    }
    acc += __half2float(d[b]) * s;
  }
  acc = warp_reduce(acc);
  if (lane == 0) y[row] = acc;
}

// ================================================ the missing factorial cells
//
// v1/v3 vary the weight layout while every lane gathers its own slice of x.
// v2 varies where x lives while the weight layout stays AoS. Neither isolates
// the two effects, so the grid below is completed: {AoS, SoA} x {x global,
// x shared}, for both formats. Every cell computes the same numbers.

// q8_0, SoA weights, x staged in shared memory.
__global__ void qgemv_q8_v5_soa_smem(const int8_t* __restrict__ Q,
                                     const __half* __restrict__ D,
                                     const float* __restrict__ x,
                                     float* __restrict__ y, int M, int K) {
  extern __shared__ float sx[];
  for (int i = threadIdx.x; i < K; i += blockDim.x) sx[i] = x[i];
  __syncthreads();
  const int warps = blockDim.x / 32;
  const int lane = threadIdx.x & 31;
  const int row = blockIdx.x * warps + (threadIdx.x >> 5);
  if (row >= M) return;
  const int nb = K / QK;
  const int4* q4 = reinterpret_cast<const int4*>(Q + (size_t)row * K);
  const __half* d = D + (size_t)row * nb;
  const int nvec = K / 16;
  float acc = 0.f;
  for (int v = lane; v < nvec; v += 32) {
    int4 raw = q4[v];
    const int8_t* q = reinterpret_cast<const int8_t*>(&raw);
    const int base = v * 16;
    float s = 0.f;
#pragma unroll
    for (int j = 0; j < 16; ++j) s += (float)q[j] * sx[base + j];
    acc += __half2float(d[base / QK]) * s;
  }
  acc = warp_reduce(acc);
  if (lane == 0) y[row] = acc;
}

// q4_0, ggml AoS weights, x staged in shared memory.
__global__ void qgemv_q4_v2_smem_x(const block_q4_0* __restrict__ W,
                                   const float* __restrict__ x,
                                   float* __restrict__ y, int M, int K) {
  extern __shared__ float sx[];
  for (int i = threadIdx.x; i < K; i += blockDim.x) sx[i] = x[i];
  __syncthreads();
  const int warps = blockDim.x / 32;
  const int lane = threadIdx.x & 31;
  const int row = blockIdx.x * warps + (threadIdx.x >> 5);
  if (row >= M) return;
  const int nb = K / QK;
  const block_q4_0* w = W + (size_t)row * nb;
  float acc = 0.f;
  for (int b = lane; b < nb; b += 32) {
    const block_q4_0 blk = w[b];
    float s = 0.f;
#pragma unroll
    for (int j = 0; j < QK / 2; ++j) {
      const uint8_t p = blk.qs[j];
      s += (float)((int)(p & 0x0F) - 8) * sx[b * QK + j];
      s += (float)((int)(p >> 4) - 8) * sx[b * QK + j + 16];
    }
    acc += __half2float(blk.d) * s;
  }
  acc = warp_reduce(acc);
  if (lane == 0) y[row] = acc;
}

// q4_0, SoA weights, x staged in shared memory.
__global__ void qgemv_q4_v5_soa_smem(const uint8_t* __restrict__ Q,
                                     const __half* __restrict__ D,
                                     const float* __restrict__ x,
                                     float* __restrict__ y, int M, int K) {
  extern __shared__ float sx[];
  for (int i = threadIdx.x; i < K; i += blockDim.x) sx[i] = x[i];
  __syncthreads();
  const int warps = blockDim.x / 32;
  const int lane = threadIdx.x & 31;
  const int row = blockIdx.x * warps + (threadIdx.x >> 5);
  if (row >= M) return;
  const int nb = K / QK;
  const int4* q4 = reinterpret_cast<const int4*>(Q + (size_t)row * (K / 2));
  const __half* d = D + (size_t)row * nb;
  float acc = 0.f;
  for (int b = lane; b < nb; b += 32) {
    int4 raw = q4[b];
    const uint8_t* p = reinterpret_cast<const uint8_t*>(&raw);
    const int base = b * QK;
    float s = 0.f;
#pragma unroll
    for (int j = 0; j < QK / 2; ++j) {
      s += (float)((int)(p[j] & 0x0F) - 8) * sx[base + j];
      s += (float)((int)(p[j] >> 4) - 8) * sx[base + j + 16];
    }
    acc += __half2float(d[b]) * s;
  }
  acc = warp_reduce(acc);
  if (lane == 0) y[row] = acc;
}

// ============================================== attacking the q4_0 unpack cost
//
// Finding 3 says q4_0 loses to q8_0 because unpacking costs more than the bytes
// save. If that is right, cutting the unpack arithmetic should narrow the gap,
// and if it does not, the claim is wrong. v6 is that test.
//
// Two changes, both exact rearrangements rather than approximations:
//
//   1. The bias comes out of the inner loop by algebra. The block sum is
//      sum((q - 8) * x) = sum(q * x) - 8 * sum(x), and sum(x) over a block does
//      not depend on the row, so it is computed once per block and reused by
//      every row. That removes one integer subtract per weight.
//   2. Nibbles are extracted four bytes at a time with a 32-bit mask instead of
//      one byte at a time, which removes three quarters of the mask and shift
//      operations.
//
// Numerics: sum(q*x) with q in [0,15] against 8*sum(x) are comparable in
// magnitude, so the cancellation is mild. The test suite checks it rather than
// assuming it.
__global__ void qgemv_q4_v6_fastunpack(const uint8_t* __restrict__ Q,
                                       const __half* __restrict__ D,
                                       const float* __restrict__ x,
                                       const float* __restrict__ xsum,
                                       float* __restrict__ y, int M, int K) {
  extern __shared__ float smem[];
  const int nb = K / QK;
  float* sx = smem;              // K floats: the activation vector
  float* sxs = smem + K;         // nb floats: per-block sums of x
  for (int i = threadIdx.x; i < K; i += blockDim.x) sx[i] = x[i];
  for (int i = threadIdx.x; i < nb; i += blockDim.x) sxs[i] = xsum[i];
  __syncthreads();

  const int warps = blockDim.x / 32;
  const int lane = threadIdx.x & 31;
  const int row = blockIdx.x * warps + (threadIdx.x >> 5);
  if (row >= M) return;
  const int4* q4 = reinterpret_cast<const int4*>(Q + (size_t)row * (K / 2));
  const __half* d = D + (size_t)row * nb;
  float acc = 0.f;
  for (int b = lane; b < nb; b += 32) {
    int4 raw = q4[b];
    const uint32_t* w = reinterpret_cast<const uint32_t*>(&raw);
    const int base = b * QK;
    float s = 0.f;
#pragma unroll
    for (int g = 0; g < 4; ++g) {          // 4 groups of 4 bytes
      const uint32_t v = w[g];
      const uint32_t lo = v & 0x0F0F0F0Fu;  // four low nibbles at once
      const uint32_t hi = (v >> 4) & 0x0F0F0F0Fu;
#pragma unroll
      for (int j = 0; j < 4; ++j) {
        const int k = g * 4 + j;
        s += (float)((lo >> (8 * j)) & 0xFFu) * sx[base + k];
        s += (float)((hi >> (8 * j)) & 0xFFu) * sx[base + k + 16];
      }
    }
    acc += __half2float(d[b]) * (s - 8.0f * sxs[b]);
  }
  acc = warp_reduce(acc);
  if (lane == 0) y[row] = acc;
}
'''
(SRC / 'qgemv_kernels.cuh').write_text(KERNELS)
print('kernels', len(KERNELS), 'bytes')

In [ ]:
BENCH = r'''// Harness: correctness first, then timing, then achieved bandwidth.
//
// Nothing here reports a number that was not measured on the device it names.
// Timing uses CUDA events around a warmed-up loop and reports a distribution,
// because a single sample of a GPU kernel is not a measurement.

#include "qgemv_kernels.cuh"

#include <cuda_runtime.h>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <functional>
#include <string>
#include <vector>

#define CUDA_OK(call)                                                          \
  do {                                                                         \
    cudaError_t e_ = (call);                                                   \
    if (e_ != cudaSuccess) {                                                   \
      fprintf(stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(e_),      \
              __FILE__, __LINE__);                                             \
      exit(1);                                                                 \
    }                                                                          \
  } while (0)

// ---------------------------------------------------------------- quantize
// Both routines follow ggml semantics so the bytes are the bytes a GGUF holds.

static void quantize_q8_0(const float* src, int n, std::vector<block_q8_0>& out) {
  const int nb = n / QK;
  out.resize(nb);
  for (int b = 0; b < nb; ++b) {
    const float* v = src + b * QK;
    float amax = 0.f;
    for (int j = 0; j < QK; ++j) amax = std::max(amax, std::fabs(v[j]));
    const float d = amax / 127.f;
    const float id = d ? 1.f / d : 0.f;
    out[b].d = __float2half(d);
    for (int j = 0; j < QK; ++j) {
      int q = (int)lrintf(v[j] * id);
      out[b].qs[j] = (int8_t)std::max(-127, std::min(127, q));
    }
  }
}

static void quantize_q4_0(const float* src, int n, std::vector<block_q4_0>& out) {
  const int nb = n / QK;
  out.resize(nb);
  for (int b = 0; b < nb; ++b) {
    const float* v = src + b * QK;
    float amax = 0.f, mx = 0.f;
    for (int j = 0; j < QK; ++j) {
      if (std::fabs(v[j]) > amax) { amax = std::fabs(v[j]); mx = v[j]; }
    }
    const float d = mx / -8.f;
    const float id = d ? 1.f / d : 0.f;
    out[b].d = __float2half(d);
    for (int j = 0; j < QK / 2; ++j) {
      int q0 = (int)(v[j] * id + 8.5f);
      int q1 = (int)(v[j + 16] * id + 8.5f);
      q0 = std::max(0, std::min(15, q0));
      q1 = std::max(0, std::min(15, q1));
      out[b].qs[j] = (uint8_t)(q0 | (q1 << 4));
    }
  }
}

// ---------------------------------------------------------------- reference
// Double-precision CPU reference over the dequantized values. This is the
// ground truth for every kernel, and it is deliberately the slow obvious code.

static void ref_q8(const std::vector<block_q8_0>& W, const std::vector<float>& x,
                   int M, int K, std::vector<double>& y) {
  const int nb = K / QK;
  y.assign(M, 0.0);
  for (int r = 0; r < M; ++r) {
    double acc = 0.0;
    const block_q8_0* w = W.data() + (size_t)r * nb;
    for (int b = 0; b < nb; ++b) {
      const double d = (double)__half2float(w[b].d);
      double s = 0.0;
      for (int j = 0; j < QK; ++j) s += (double)w[b].qs[j] * (double)x[b * QK + j];
      acc += d * s;
    }
    y[r] = acc;
  }
}

static void ref_q4(const std::vector<block_q4_0>& W, const std::vector<float>& x,
                   int M, int K, std::vector<double>& y) {
  const int nb = K / QK;
  y.assign(M, 0.0);
  for (int r = 0; r < M; ++r) {
    double acc = 0.0;
    const block_q4_0* w = W.data() + (size_t)r * nb;
    for (int b = 0; b < nb; ++b) {
      const double d = (double)__half2float(w[b].d);
      double s = 0.0;
      for (int j = 0; j < QK / 2; ++j) {
        const uint8_t p = w[b].qs[j];
        s += (double)((int)(p & 0x0F) - 8) * (double)x[b * QK + j];
        s += (double)((int)(p >> 4) - 8) * (double)x[b * QK + j + 16];
      }
      acc += d * s;
    }
    y[r] = acc;
  }
}

// ---------------------------------------------------------------- timing

struct Stats { double median, mn, mx, mean, sd, p95; };

static Stats summarize(std::vector<float> ms) {
  std::sort(ms.begin(), ms.end());
  Stats s{};
  const size_t n = ms.size();
  s.median = ms[n / 2];
  s.mn = ms.front();
  s.mx = ms.back();
  s.p95 = ms[(size_t)(0.95 * (n - 1))];
  double sum = 0.0;
  for (float v : ms) sum += v;
  s.mean = sum / n;
  double var = 0.0;
  for (float v : ms) var += (v - s.mean) * (v - s.mean);
  s.sd = std::sqrt(var / n);
  return s;
}

template <typename F>
static Stats time_kernel(F launch, int warmup, int reps) {
  cudaEvent_t a, b;
  CUDA_OK(cudaEventCreate(&a));
  CUDA_OK(cudaEventCreate(&b));
  for (int i = 0; i < warmup; ++i) launch();
  CUDA_OK(cudaDeviceSynchronize());
  std::vector<float> ms;
  ms.reserve(reps);
  for (int i = 0; i < reps; ++i) {
    CUDA_OK(cudaEventRecord(a));
    launch();
    CUDA_OK(cudaEventRecord(b));
    CUDA_OK(cudaEventSynchronize(b));
    float t = 0.f;
    CUDA_OK(cudaEventElapsedTime(&t, a, b));
    ms.push_back(t);
  }
  CUDA_OK(cudaGetLastError());
  cudaEventDestroy(a);
  cudaEventDestroy(b);
  return summarize(ms);
}

// ---------------------------------------------------------------- ceiling
// A pure streaming read. This is the measured upper bound that every achieved
// figure below is a fraction of. The datasheet number is not that bound.

__global__ void stream_read(const int4* __restrict__ src, size_t n4,
                            float* __restrict__ sink) {
  size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
  const size_t stride = (size_t)gridDim.x * blockDim.x;
  int4 acc = make_int4(0, 0, 0, 0);
  for (; i < n4; i += stride) {
    int4 v = src[i];
    acc.x ^= v.x; acc.y ^= v.y; acc.z ^= v.z; acc.w ^= v.w;
  }
  if ((acc.x | acc.y | acc.z | acc.w) == 0x7fffffff) sink[0] = 1.f;  // never taken
}

static void run_ceiling(size_t bytes, int reps) {
  const size_t n4 = bytes / sizeof(int4);
  int4* d = nullptr;
  float* sink = nullptr;
  CUDA_OK(cudaMalloc(&d, n4 * sizeof(int4)));
  CUDA_OK(cudaMalloc(&sink, sizeof(float)));
  CUDA_OK(cudaMemset(d, 1, n4 * sizeof(int4)));
  int dev = 0;
  cudaDeviceProp p{};
  CUDA_OK(cudaGetDeviceProperties(&p, dev));
  const int threads = 256;
  const int blocks = p.multiProcessorCount * 16;
  Stats s = time_kernel([&] { stream_read<<<blocks, threads>>>(d, n4, sink); }, 10, reps);
  const double gb = (double)(n4 * sizeof(int4)) / 1e9;
  printf("{\"kernel\":\"stream_read\",\"bytes\":%zu,\"ms_median\":%.6f,"
         "\"ms_min\":%.6f,\"ms_sd\":%.6f,\"gbps_median\":%.3f,\"gbps_max\":%.3f}\n",
         n4 * sizeof(int4), s.median, s.mn, s.sd, gb / (s.median / 1e3),
         gb / (s.mn / 1e3));

  // D2D copy, a second independent read of the same ceiling.
  int4* d2 = nullptr;
  CUDA_OK(cudaMalloc(&d2, n4 * sizeof(int4)));
  Stats c = time_kernel(
      [&] { cudaMemcpyAsync(d2, d, n4 * sizeof(int4), cudaMemcpyDeviceToDevice); }, 10,
      reps);
  const double gb2 = 2.0 * gb;  // read + write
  printf("{\"kernel\":\"memcpy_d2d\",\"bytes\":%zu,\"ms_median\":%.6f,"
         "\"gbps_median\":%.3f}\n",
         2 * n4 * sizeof(int4), c.median, gb2 / (c.median / 1e3));
  cudaFree(d);
  cudaFree(d2);
  cudaFree(sink);
}

// ---------------------------------------------------------------- main

struct Result {
  const char* name;
  const char* fmt;
  const char* layout;
  Stats s;
  double gbps;
  double max_abs_err;
  double max_rel_err;
};

static void emit(const Result& r, size_t wbytes, int M, int K) {
  printf("{\"kernel\":\"%s\",\"format\":\"%s\",\"layout\":\"%s\",\"M\":%d,\"K\":%d,"
         "\"weight_bytes\":%zu,\"ms_median\":%.6f,\"ms_min\":%.6f,\"ms_p95\":%.6f,"
         "\"ms_sd\":%.6f,\"gbps\":%.3f,\"max_abs_err\":%.6g,\"max_rel_err\":%.6g}\n",
         r.name, r.fmt, r.layout, M, K, wbytes, r.s.median, r.s.mn, r.s.p95, r.s.sd,
         r.gbps, r.max_abs_err, r.max_rel_err);
  fflush(stdout);
}

static void check(const std::vector<float>& got, const std::vector<double>& want,
                  double& max_abs, double& max_rel) {
  max_abs = 0.0;
  max_rel = 0.0;
  double scale = 0.0;
  for (double v : want) scale = std::max(scale, std::fabs(v));
  for (size_t i = 0; i < want.size(); ++i) {
    const double e = std::fabs((double)got[i] - want[i]);
    max_abs = std::max(max_abs, e);
    if (scale > 0) max_rel = std::max(max_rel, e / scale);
  }
}

int main(int argc, char** argv) {
  int M = 4096, K = 4096, reps = 200, warmup = 20;
  std::string mode = "bench";
  double sustain_s = 0.0;
  std::string only;
  for (int i = 1; i < argc; ++i) {
    std::string a = argv[i];
    auto next = [&]() { return std::string(argv[++i]); };
    if (a == "--M") M = std::stoi(next());
    else if (a == "--K") K = std::stoi(next());
    else if (a == "--reps") reps = std::stoi(next());
    else if (a == "--mode") mode = next();
    else if (a == "--sustain") sustain_s = std::stod(next());
    else if (a == "--only") only = next();
  }
  if (K % 512 != 0) { fprintf(stderr, "K must be a multiple of 512\n"); return 1; }

  cudaDeviceProp p{};
  CUDA_OK(cudaGetDeviceProperties(&p, 0));
  int memclk_khz = 0, buswidth = 0, smclk_khz = 0;
  cudaDeviceGetAttribute(&memclk_khz, cudaDevAttrMemoryClockRate, 0);
  cudaDeviceGetAttribute(&buswidth, cudaDevAttrGlobalMemoryBusWidth, 0);
  cudaDeviceGetAttribute(&smclk_khz, cudaDevAttrClockRate, 0);
  printf("{\"device\":\"%s\",\"cc\":\"sm_%d%d\",\"sms\":%d,\"l2_bytes\":%d,"
         "\"mem_clock_khz\":%d,\"bus_bits\":%d,\"sm_clock_khz\":%d,"
         "\"datasheet_gbps_x2\":%.2f}\n",
         p.name, p.major, p.minor, p.multiProcessorCount, p.l2CacheSize, memclk_khz,
         buswidth, smclk_khz, 2.0 * memclk_khz * 1e3 * (buswidth / 8.0) / 1e9);
  fflush(stdout);

  if (mode == "ceiling") {
    size_t bytes = 512u * 1024u * 1024u;
    run_ceiling(bytes, reps);
    return 0;
  }

  const int nb = K / QK;
  const size_t nw = (size_t)M * K;

  // Host data. Weights are drawn once and quantized once; every kernel below
  // sees the same numbers, so any output difference is a kernel bug.
  std::vector<float> hw(nw), hx(K);
  srand(1234);
  for (size_t i = 0; i < nw; ++i) hw[i] = (float)((rand() / (double)RAND_MAX) * 2.0 - 1.0);
  for (int i = 0; i < K; ++i) hx[i] = (float)((rand() / (double)RAND_MAX) * 2.0 - 1.0);

  std::vector<block_q8_0> q8(( size_t)M * nb);
  std::vector<block_q4_0> q4((size_t)M * nb);
  for (int r = 0; r < M; ++r) {
    std::vector<block_q8_0> t8;
    std::vector<block_q4_0> t4;
    quantize_q8_0(hw.data() + (size_t)r * K, K, t8);
    quantize_q4_0(hw.data() + (size_t)r * K, K, t4);
    memcpy(&q8[(size_t)r * nb], t8.data(), nb * sizeof(block_q8_0));
    memcpy(&q4[(size_t)r * nb], t4.data(), nb * sizeof(block_q4_0));
  }

  // SoA planes carry the identical quantized values, only rearranged.
  std::vector<int8_t> soa8_q(nw);
  std::vector<__half> soa8_d((size_t)M * nb);
  std::vector<uint8_t> soa4_q(nw / 2);
  std::vector<__half> soa4_d((size_t)M * nb);
  for (int r = 0; r < M; ++r) {
    for (int b = 0; b < nb; ++b) {
      const block_q8_0& B8 = q8[(size_t)r * nb + b];
      soa8_d[(size_t)r * nb + b] = B8.d;
      memcpy(&soa8_q[(size_t)r * K + (size_t)b * QK], B8.qs, QK);
      const block_q4_0& B4 = q4[(size_t)r * nb + b];
      soa4_d[(size_t)r * nb + b] = B4.d;
      memcpy(&soa4_q[(size_t)r * (K / 2) + (size_t)b * (QK / 2)], B4.qs, QK / 2);
    }
  }

  std::vector<double> ry8, ry4;
  ref_q8(q8, hx, M, K, ry8);
  ref_q4(q4, hx, M, K, ry4);

  // Device
  block_q8_0* dq8 = nullptr; block_q4_0* dq4 = nullptr;
  int8_t* ds8q = nullptr; uint8_t* ds4q = nullptr;
  __half *ds8d = nullptr, *ds4d = nullptr;
  float *dx = nullptr, *dy = nullptr, *dxsum = nullptr;
  CUDA_OK(cudaMalloc(&dq8, q8.size() * sizeof(block_q8_0)));
  CUDA_OK(cudaMalloc(&dq4, q4.size() * sizeof(block_q4_0)));
  CUDA_OK(cudaMalloc(&ds8q, soa8_q.size()));
  CUDA_OK(cudaMalloc(&ds4q, soa4_q.size()));
  CUDA_OK(cudaMalloc(&ds8d, soa8_d.size() * sizeof(__half)));
  CUDA_OK(cudaMalloc(&ds4d, soa4_d.size() * sizeof(__half)));
  CUDA_OK(cudaMalloc(&dx, K * sizeof(float)));
  CUDA_OK(cudaMalloc(&dy, M * sizeof(float)));
  CUDA_OK(cudaMalloc(&dxsum, nb * sizeof(float)));
  CUDA_OK(cudaMemcpy(dq8, q8.data(), q8.size() * sizeof(block_q8_0), cudaMemcpyHostToDevice));
  CUDA_OK(cudaMemcpy(dq4, q4.data(), q4.size() * sizeof(block_q4_0), cudaMemcpyHostToDevice));
  CUDA_OK(cudaMemcpy(ds8q, soa8_q.data(), soa8_q.size(), cudaMemcpyHostToDevice));
  CUDA_OK(cudaMemcpy(ds4q, soa4_q.data(), soa4_q.size(), cudaMemcpyHostToDevice));
  CUDA_OK(cudaMemcpy(ds8d, soa8_d.data(), soa8_d.size() * sizeof(__half), cudaMemcpyHostToDevice));
  CUDA_OK(cudaMemcpy(ds4d, soa4_d.data(), soa4_d.size() * sizeof(__half), cudaMemcpyHostToDevice));
  CUDA_OK(cudaMemcpy(dx, hx.data(), K * sizeof(float), cudaMemcpyHostToDevice));
  // Per-block sums of x, used by q4_v6 to lift the -8 bias out of the inner
  // loop. Computed once on the host; it is nb floats and row independent.
  std::vector<float> hxsum(nb, 0.f);
  for (int b = 0; b < nb; ++b) {
    float t = 0.f;
    for (int j = 0; j < QK; ++j) t += hx[b * QK + j];
    hxsum[b] = t;
  }
  CUDA_OK(cudaMemcpy(dxsum, hxsum.data(), nb * sizeof(float), cudaMemcpyHostToDevice));

  const size_t wb8 = q8.size() * sizeof(block_q8_0);
  const size_t wb4 = q4.size() * sizeof(block_q4_0);
  std::vector<float> hy(M);

  auto run_one = [&](const char* name, const char* fmt, const char* layout,
                     size_t wbytes, const std::vector<double>& ref,
                     std::function<void()> launch) {
    if (!only.empty() && only != name) return;
    CUDA_OK(cudaMemset(dy, 0, M * sizeof(float)));
    launch();
    CUDA_OK(cudaDeviceSynchronize());
    CUDA_OK(cudaGetLastError());
    CUDA_OK(cudaMemcpy(hy.data(), dy, M * sizeof(float), cudaMemcpyDeviceToHost));
    Result r{name, fmt, layout, {}, 0, 0, 0};
    check(hy, ref, r.max_abs_err, r.max_rel_err);
    if (sustain_s > 0.0) {
      // Sustained mode: run flat out for a wall-clock window so an external
      // energy counter has something to integrate over.
      double elapsed = 0.0;
      cudaEvent_t a, b;
      cudaEventCreate(&a); cudaEventCreate(&b);
      cudaEventRecord(a);
      long long iters = 0;
      while (elapsed < sustain_s * 1000.0) {
        for (int i = 0; i < 200; ++i) { launch(); ++iters; }
        CUDA_OK(cudaDeviceSynchronize());
        cudaEventRecord(b); cudaEventSynchronize(b);
        float t; cudaEventElapsedTime(&t, a, b); elapsed = t;
      }
      printf("{\"sustain\":\"%s\",\"iters\":%lld,\"ms\":%.3f,\"bytes_total\":%.6g}\n",
             name, iters, elapsed, (double)wbytes * (double)iters);
      fflush(stdout);
      return;
    }
    r.s = time_kernel(launch, warmup, reps);
    r.gbps = ((double)wbytes / 1e9) / (r.s.median / 1e3);
    emit(r, wbytes, M, K);
  };

  const int TPB = 256;
  const int warps_per_block = TPB / 32;

  run_one("q8_v0_naive", "q8_0", "aos", wb8, ry8, [&] {
    qgemv_q8_v0_naive<<<(M + TPB - 1) / TPB, TPB>>>(dq8, dx, dy, M, K);
  });
  run_one("q8_v1_warp", "q8_0", "aos", wb8, ry8, [&] {
    qgemv_q8_v1_warp<<<(M + warps_per_block - 1) / warps_per_block, TPB>>>(dq8, dx, dy, M, K);
  });
  run_one("q8_v2_smem_x", "q8_0", "aos", wb8, ry8, [&] {
    qgemv_q8_v2_smem_x<<<(M + warps_per_block - 1) / warps_per_block, TPB,
                         K * sizeof(float)>>>(dq8, dx, dy, M, K);
  });
  run_one("q8_v3_soa_vec", "q8_0", "soa", wb8, ry8, [&] {
    qgemv_q8_v3_soa_vec<<<(M + warps_per_block - 1) / warps_per_block, TPB>>>(ds8q, ds8d, dx, dy, M, K);
  });
  run_one("q8_v4_split2", "q8_0", "soa", wb8, ry8, [&] {
    const int rpb = TPB / (32 * 2);
    qgemv_q8_v4_split<2><<<(M + rpb - 1) / rpb, TPB, warps_per_block * sizeof(float)>>>(
        ds8q, ds8d, dx, dy, M, K);
  });
  run_one("q8_v4_split4", "q8_0", "soa", wb8, ry8, [&] {
    const int rpb = TPB / (32 * 4);
    qgemv_q8_v4_split<4><<<(M + rpb - 1) / rpb, TPB, warps_per_block * sizeof(float)>>>(
        ds8q, ds8d, dx, dy, M, K);
  });
  run_one("q8_v5_soa_smem", "q8_0", "soa", wb8, ry8, [&] {
    qgemv_q8_v5_soa_smem<<<(M + warps_per_block - 1) / warps_per_block, TPB,
                           K * sizeof(float)>>>(ds8q, ds8d, dx, dy, M, K);
  });
  run_one("q4_v0_naive", "q4_0", "aos", wb4, ry4, [&] {
    qgemv_q4_v0_naive<<<(M + TPB - 1) / TPB, TPB>>>(dq4, dx, dy, M, K);
  });
  run_one("q4_v1_warp", "q4_0", "aos", wb4, ry4, [&] {
    qgemv_q4_v1_warp<<<(M + warps_per_block - 1) / warps_per_block, TPB>>>(dq4, dx, dy, M, K);
  });
  run_one("q4_v2_smem_x", "q4_0", "aos", wb4, ry4, [&] {
    qgemv_q4_v2_smem_x<<<(M + warps_per_block - 1) / warps_per_block, TPB,
                         K * sizeof(float)>>>(dq4, dx, dy, M, K);
  });
  run_one("q4_v3_soa_vec", "q4_0", "soa", wb4, ry4, [&] {
    qgemv_q4_v3_soa_vec<<<(M + warps_per_block - 1) / warps_per_block, TPB>>>(ds4q, ds4d, dx, dy, M, K);
  });
  run_one("q4_v5_soa_smem", "q4_0", "soa", wb4, ry4, [&] {
    qgemv_q4_v5_soa_smem<<<(M + warps_per_block - 1) / warps_per_block, TPB,
                           K * sizeof(float)>>>(ds4q, ds4d, dx, dy, M, K);
  });
  run_one("q4_v6_fastunpack", "q4_0", "soa", wb4, ry4, [&] {
    qgemv_q4_v6_fastunpack<<<(M + warps_per_block - 1) / warps_per_block, TPB,
                             (K + nb) * sizeof(float)>>>(ds4q, ds4d, dx, dxsum, dy, M, K);
  });

  cudaFree(dq8); cudaFree(dq4); cudaFree(ds8q); cudaFree(ds4q);
  cudaFree(ds8d); cudaFree(ds4d); cudaFree(dx); cudaFree(dy); cudaFree(dxsum);
  return 0;
}
'''
(SRC / 'bench.cu').write_text(BENCH)
print('bench', len(BENCH), 'bytes')

In [ ]:
# Detect the architecture and build for it, rather than assuming one.
import torch
try:
    cc = torch.cuda.get_device_capability(0)
    arch = f"sm_{cc[0]}{cc[1]}"
except Exception:
    arch = "sm_75"
print("building for", arch)
r = subprocess.run(
    ["nvcc", "-O3", "-std=c++17", f"-arch={arch}", "-lineinfo",
     "-I", str(SRC), str(SRC / "bench.cu"), "-o", str(SRC / "bench")],
    capture_output=True, text=True)
print(r.returncode)
print(r.stdout[-3000:]); print(r.stderr[-3000:])

In [ ]:
def run(args):
    r = subprocess.run([str(SRC / "bench")] + args, capture_output=True, text=True)
    if r.returncode != 0:
        print("STDERR:", r.stderr[-2000:])
    out = []
    for line in r.stdout.splitlines():
        line = line.strip()
        if line.startswith("{"):
            rec = json.loads(line)
            out.append(rec)
            print(json.dumps(rec))
    return out

results = {}
print("=== measured bandwidth ceiling ===")
results["ceiling"] = run(["--mode", "ceiling", "--reps", "50"])
print()

# Pick a shape whose weight matrix is far larger than this L2, or the timing
# loop measures cache bandwidth and calls it memory bandwidth. The first pass
# of this study got that wrong on the 48 MiB L2 parts and reported a kernel
# running at 274% of its own measured DRAM roof, which is impossible and was
# the tell. K is fixed at 4096; M is whatever makes the weights 8x the L2.
K = 4096
l2 = torch.cuda.get_device_properties(0).L2_cache_size
bytes_per_row_q8 = K * 34 / 32
M_big = int(8 * l2 / bytes_per_row_q8)
M_big = max(11008, (M_big // 32) * 32)
free_b, total_b = torch.cuda.mem_get_info()
# The harness holds q8 AoS + q4 AoS + both SoA planes at once, about 2.7x the
# q8 weight bytes, so keep well inside what is free.
while M_big > 11008 and M_big * bytes_per_row_q8 * 2.7 > 0.5 * free_b:
    M_big = (M_big // 2 // 32) * 32
print(f"L2 = {l2/1048576:.0f} MiB, free = {free_b/2**30:.1f} GiB")
print(f"large shape M = {M_big}, q8_0 weights = "
      f"{M_big*bytes_per_row_q8/1048576:.0f} MiB = "
      f"{M_big*bytes_per_row_q8/l2:.1f}x L2")

SHAPES = [(4096, K, "4096x4096"), (11008, K, "11008x4096")]
if M_big > 11008:
    SHAPES.append((M_big, K, f"{M_big}x{K}"))
results["shape_choice"] = {"l2_bytes": l2, "M_big": M_big, "K": K}

for M, Kk, tag in SHAPES:
    reps = 200 if M <= 11008 else 40
    print(f"=== ladder {tag} ({reps} reps) ===")
    results[tag] = run(["--M", str(M), "--K", str(Kk), "--reps", str(reps)])
    print()
results["big_tag"] = SHAPES[-1][2]

## Nsight Compute

This is the step that fails on the development machine, where `ncu` returns
`ERR_NVGPUCTRPERM` under WSL2.

Being root is **not** what fixes it. Kaggle has `ncu` installed and runs the
notebook as uid 0, and counters are refused there anyway, because the driver
gates counter access independently of the user and that Kaggle vGPU does not
pass it through to the guest. Colab does grant it. So this cell is the one place
in the study where the limiter is measured rather than inferred, and it only
works on one of the two free platforms.

The cell below locates `ncu`, installs Nsight Compute if the image does not ship
it, and reports plainly if counters are still refused rather than printing an
empty table.

In [ ]:
import glob, shutil

def find_ncu():
    p = shutil.which("ncu")
    if p:
        return p
    for pat in ("/opt/nvidia/nsight-compute/*/ncu",
                "/usr/local/cuda*/bin/ncu",
                "/usr/local/NVIDIA-Nsight-Compute*/ncu"):
        hits = sorted(glob.glob(pat))
        if hits:
            return hits[-1]
    return None

ncu = find_ncu()
if ncu is None:
    print("ncu not present, installing nsight-compute ...")
    subprocess.run(["apt-get", "update", "-qq"], capture_output=True, text=True)
    r = subprocess.run(["apt-get", "install", "-y", "-qq", "nsight-compute"],
                       capture_output=True, text=True)
    print("apt exit", r.returncode, r.stderr[-500:])
    ncu = find_ncu()
print("ncu:", ncu)
print("running as uid", os.getuid())

In [ ]:
ncu_metrics = ",".join([
    "gpu__time_duration.avg",
    "dram__bytes_read.sum",
    "dram__throughput.avg.pct_of_peak_sustained_elapsed",
    "sm__throughput.avg.pct_of_peak_sustained_elapsed",
    "smsp__inst_executed.sum",
    "sm__warps_active.avg.pct_of_peak_sustained_active",
    "l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum",
])

ncu_out = {}
if ncu is None:
    print("no ncu available on this image; skipping the profiler section")
else:
    Mp = results["shape_choice"]["M_big"]
    for k in ["q8_v1_warp", "q8_v3_soa_vec", "q8_v2_smem_x",
              "q8_v5_soa_smem", "q4_v5_soa_smem", "q4_v6_fastunpack"]:
        print("=" * 24, k)
        r = subprocess.run(
            [ncu, "--metrics", ncu_metrics, "--launch-count", "1",
             "--target-processes", "all", "--csv", str(SRC / "bench"),
             "--M", str(Mp), "--K", "4096", "--reps", "1", "--only", k],
            capture_output=True, text=True)
        blob = r.stdout + r.stderr
        if "ERR_NVGPUCTRPERM" in blob:
            print("counters refused on this VM (ERR_NVGPUCTRPERM). "
                  "Nothing to fix from inside the notebook.")
            break
        ncu_out[k] = r.stdout
        print(r.stdout[-3500:] or r.stderr[-1500:])
results["ncu"] = ncu_out

### Reading the profiler against the prediction

The controlled experiment on the development machine concluded that the
activation gather, not the weight stream, was the limiter for `q8_v1_warp` and
`q8_v3_soa_vec`, and that `q4_v5_soa_smem` is ALU bound rather than memory
bound. Those are falsifiable against the counters above:

- `q8_v1_warp` and `q8_v3_soa_vec` should show **low** `dram__throughput` with
  **high** `l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum`, which is the
  signature of a gather that never reaches DRAM efficiently.
- `q8_v5_soa_smem` should show the **highest** `dram__throughput` of the set.
- `q4_v5_soa_smem` should show high `sm__throughput` and `smsp__inst_executed`
  against modest `dram__throughput`, which is what "the unpacking cost more than
  the bytes saved" looks like in counters.

If those do not hold, the mechanism in the README is wrong and the README needs
changing, not the data.

## Energy

`nvmlDeviceGetTotalEnergyConsumption` is a monotonic counter, present on Volta
and newer. It is absent on the P100 (Pascal), and this cell says so rather than
substituting an integrated power estimate without labelling it. The counter is
read twice per window and never sampled, because sampling perturbs the draw on
at least one part in this study.

In [ ]:
try:
    import pynvml
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-ml-py"],
                   capture_output=True, text=True)
    import pynvml
import time
pynvml.nvmlInit()
h = pynvml.nvmlDeviceGetHandleByIndex(0)
name = pynvml.nvmlDeviceGetName(h)
name = name.decode() if isinstance(name, bytes) else name
has_counter = True
try:
    pynvml.nvmlDeviceGetTotalEnergyConsumption(h)
except pynvml.NVMLError as e:
    has_counter = False
    print(f"{name}: energy counter NOT supported ({e}). Skipping energy.")

if has_counter:
    def window(fn):
        e0 = pynvml.nvmlDeviceGetTotalEnergyConsumption(h); t0 = time.perf_counter()
        fn()
        e1 = pynvml.nvmlDeviceGetTotalEnergyConsumption(h); t1 = time.perf_counter()
        return (e1 - e0) / 1000.0, t1 - t0

    j_idle, s_idle = window(lambda: time.sleep(15))
    idle_w = j_idle / s_idle
    print(f"{name} idle {idle_w:.3f} W over {s_idle:.1f} s")
    energy = {"device": name, "idle_W": idle_w, "runs": []}
    Me = results["shape_choice"]["M_big"]
    for k in ["q8_v0_naive", "q8_v2_smem_x", "q8_v5_soa_smem",
              "q4_v5_soa_smem", "q4_v6_fastunpack"]:
        rec = {}
        def go():
            r = subprocess.run([str(SRC / "bench"), "--M", str(Me), "--K", "4096",
                                "--sustain", "20", "--only", k],
                               capture_output=True, text=True)
            for line in r.stdout.splitlines():
                if '"sustain"' in line:
                    rec.update(json.loads(line))
        j, s = window(go)
        if not rec:
            print(k, "no sustain record"); continue
        row = {"kernel": k, "joules": j, "wall_s": s, "watts": j / s,
               "iters": rec["iters"], "bytes_total": rec["bytes_total"],
               "joules_above_idle": j - idle_w * s,
               "pJ_per_weight_byte": j / rec["bytes_total"] * 1e12}
        energy["runs"].append(row)
        print(json.dumps(row))
    results["energy"] = energy

In [ ]:
TRITON = r'''"""The same quantized GEMV in Triton, plus a PyTorch reference.

Two reasons this file exists. Triton is named alongside CUDA in the job
descriptions this repository answers, and having both implementations lets the
same measurement be made twice through different compilers. The interesting
output is not which one wins, it is how far a few lines of Triton get you
against hand-written CUDA on a kernel that is supposed to be bandwidth bound.

The layout is the SoA one: a contiguous int8 quant plane and a separate fp16
scale plane. The ggml AoS block struct is 34 bytes and Triton has no good way
to address a 34-byte stride, which is itself part of the finding in the README.

Run:  python triton_qgemv.py --M 11008 --K 4096
"""

import argparse
import json
import statistics
import sys

import torch
import triton
import triton.language as tl

QK = 32


# ------------------------------------------------------------------ kernels

@triton.jit
def _qgemv_q8_soa(
    Q,        # int8 [M, K]
    D,        # fp16 [M, K // QK]
    X,        # fp32 [K]
    Y,        # fp32 [M]
    M, K,
    BLOCK: tl.constexpr,
    QK: tl.constexpr,
):
    """One program per output row. BLOCK columns per iteration."""
    row = tl.program_id(0)
    acc = tl.zeros((), dtype=tl.float32)
    for k0 in range(0, K, BLOCK):
        offs = k0 + tl.arange(0, BLOCK)
        mask = offs < K
        q = tl.load(Q + row * K + offs, mask=mask, other=0).to(tl.float32)
        x = tl.load(X + offs, mask=mask, other=0.0)
        # One scale per QK columns, gathered per element. The compiler folds
        # this to one load per block of 32 because the index is uniform there.
        d = tl.load(D + row * (K // QK) + offs // QK, mask=mask, other=0.0).to(tl.float32)
        acc += tl.sum(q * d * x, axis=0)
    tl.store(Y + row, acc)


@triton.jit
def _qgemv_q4_soa(
    Q,        # uint8 [M, K // 2], ggml nibble order: low half then high half
    D,        # fp16 [M, K // QK]
    X,        # fp32 [K]
    Y,        # fp32 [M]
    M, K,
    NBLK: tl.constexpr,   # q4_0 blocks handled per iteration
    QK: tl.constexpr,
):
    row = tl.program_id(0)
    nb = K // QK
    acc = tl.zeros((), dtype=tl.float32)
    for b0 in range(0, nb, NBLK):
        blocks = b0 + tl.arange(0, NBLK)
        bmask = blocks < nb
        # Byte offsets inside each block: 16 bytes hold 32 weights.
        half = tl.arange(0, QK // 2)
        byte_off = blocks[:, None] * (QK // 2) + half[None, :]
        p = tl.load(Q + row * (K // 2) + byte_off, mask=bmask[:, None], other=0)
        lo = (p & 0x0F).to(tl.float32) - 8.0
        hi = ((p >> 4) & 0x0F).to(tl.float32) - 8.0
        x_lo = tl.load(X + blocks[:, None] * QK + half[None, :], mask=bmask[:, None], other=0.0)
        x_hi = tl.load(X + blocks[:, None] * QK + half[None, :] + 16, mask=bmask[:, None], other=0.0)
        d = tl.load(D + row * nb + blocks, mask=bmask, other=0.0).to(tl.float32)
        partial = tl.sum(lo * x_lo + hi * x_hi, axis=1)
        acc += tl.sum(partial * d, axis=0)
    tl.store(Y + row, acc)


# ------------------------------------------------------------ quantization

def quantize_q8_0(W):
    """ggml q8_0: per 32-weight block, d = amax/127, q = round(w/d)."""
    M, K = W.shape
    Wb = W.reshape(M, K // QK, QK)
    amax = Wb.abs().amax(dim=2)
    d = amax / 127.0
    idv = torch.where(d > 0, 1.0 / d, torch.zeros_like(d))
    q = torch.round(Wb * idv[:, :, None]).clamp(-127, 127).to(torch.int8)
    return q.reshape(M, K).contiguous(), d.to(torch.float16).contiguous()


def quantize_q4_0(W):
    """ggml q4_0: d = (signed value of largest magnitude) / -8, q = round(w/d)+8."""
    M, K = W.shape
    Wb = W.reshape(M, K // QK, QK)
    idx = Wb.abs().argmax(dim=2, keepdim=True)
    mx = torch.gather(Wb, 2, idx).squeeze(2)
    d = mx / -8.0
    idv = torch.where(d != 0, 1.0 / d, torch.zeros_like(d))
    q = torch.clamp((Wb * idv[:, :, None] + 8.5).to(torch.int32), 0, 15)
    lo = q[:, :, :16]
    hi = q[:, :, 16:]
    packed = (lo | (hi << 4)).to(torch.uint8)
    return packed.reshape(M, K // 2).contiguous(), d.to(torch.float16).contiguous()


def dequant_q8_0(q, d):
    M, K = q.shape
    return (q.reshape(M, K // QK, QK).to(torch.float32)
            * d.to(torch.float32)[:, :, None]).reshape(M, K)


def dequant_q4_0(packed, d):
    M, Kh = packed.shape
    K = Kh * 2
    p = packed.reshape(M, K // QK, QK // 2).to(torch.int32)
    lo = (p & 0x0F).to(torch.float32) - 8.0
    hi = ((p >> 4) & 0x0F).to(torch.float32) - 8.0
    w = torch.cat([lo, hi], dim=2)
    return (w * d.to(torch.float32)[:, :, None]).reshape(M, K)


# ------------------------------------------------------------------ timing

def time_ms(fn, warmup=20, reps=200):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    a = torch.cuda.Event(enable_timing=True)
    b = torch.cuda.Event(enable_timing=True)
    for _ in range(reps):
        a.record()
        fn()
        b.record()
        b.synchronize()
        samples.append(a.elapsed_time(b))
    samples.sort()
    return {
        "median": samples[len(samples) // 2],
        "min": samples[0],
        "p95": samples[int(0.95 * (len(samples) - 1))],
        "sd": statistics.pstdev(samples),
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--M", type=int, default=11008)
    ap.add_argument("--K", type=int, default=4096)
    ap.add_argument("--reps", type=int, default=200)
    ap.add_argument("--block", type=int, default=1024)
    args = ap.parse_args()

    dev = "cuda"
    torch.manual_seed(1234)
    M, K = args.M, args.K
    W = (torch.rand(M, K, device=dev) * 2 - 1)
    x = (torch.rand(K, device=dev) * 2 - 1)

    p = torch.cuda.get_device_properties(0)
    print(json.dumps({
        "device": p.name, "cc": f"sm_{p.major}{p.minor}",
        "sms": p.multi_processor_count, "torch": torch.__version__,
        "triton": triton.__version__, "M": M, "K": K,
    }))

    results = []

    # ---- q8_0
    q8, d8 = quantize_q8_0(W)
    ref8 = dequant_q8_0(q8, d8) @ x          # PyTorch reference, fp32
    y = torch.empty(M, device=dev, dtype=torch.float32)

    def run8():
        _qgemv_q8_soa[(M,)](q8, d8, x, y, M, K, BLOCK=args.block, QK=QK)

    run8()
    torch.cuda.synchronize()
    err8 = (y - ref8).abs().max().item()
    rel8 = err8 / ref8.abs().max().item()
    t8 = time_ms(run8, reps=args.reps)
    wb8 = M * K * 34 / 32                     # same byte count the CUDA harness uses
    results.append({"kernel": "triton_q8_soa", "format": "q8_0",
                    "ms_median": t8["median"], "ms_min": t8["min"], "ms_sd": t8["sd"],
                    "gbps": (wb8 / 1e9) / (t8["median"] / 1e3),
                    "max_abs_err": err8, "max_rel_err": rel8})

    # ---- q4_0
    q4, d4 = quantize_q4_0(W)
    ref4 = dequant_q4_0(q4, d4) @ x
    y4 = torch.empty(M, device=dev, dtype=torch.float32)

    def run4():
        _qgemv_q4_soa[(M,)](q4, d4, x, y4, M, K, NBLK=16, QK=QK)

    run4()
    torch.cuda.synchronize()
    err4 = (y4 - ref4).abs().max().item()
    rel4 = err4 / ref4.abs().max().item()
    t4 = time_ms(run4, reps=args.reps)
    wb4 = M * K * 18 / 32
    results.append({"kernel": "triton_q4_soa", "format": "q4_0",
                    "ms_median": t4["median"], "ms_min": t4["min"], "ms_sd": t4["sd"],
                    "gbps": (wb4 / 1e9) / (t4["median"] / 1e3),
                    "max_abs_err": err4, "max_rel_err": rel4})

    # ---- fp16 cuBLAS matvec, for scale. Not a competitor: it reads 2 bytes per
    # weight instead of 1.06 or 0.56, so it is here to show what the same shape
    # costs when nothing is quantized, and to sanity check the harness.
    Wh = W.to(torch.float16)
    xh = x.to(torch.float16)
    yh = torch.empty(M, device=dev, dtype=torch.float16)

    def runh():
        torch.mv(Wh, xh, out=yh)

    th = time_ms(runh, reps=args.reps)
    results.append({"kernel": "torch_fp16_mv", "format": "fp16",
                    "ms_median": th["median"], "ms_min": th["min"], "ms_sd": th["sd"],
                    "gbps": (M * K * 2 / 1e9) / (th["median"] / 1e3),
                    "max_abs_err": None, "max_rel_err": None})

    for r in results:
        print(json.dumps(r))


if __name__ == "__main__":
    main()
'''
(SRC / 'triton_qgemv.py').write_text(TRITON)
r = subprocess.run([sys.executable, str(SRC / 'triton_qgemv.py'),
                    '--M','11008','--K','4096'], capture_output=True, text=True)
print(r.stdout[-4000:]); print(r.stderr[-3000:])

In [ ]:
out = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").exists() else pathlib.Path(".")
dest = out / OUT_NAME
dest.write_text(json.dumps(results, indent=2))
print("wrote", dest, dest.stat().st_size, "bytes")

# On Colab, hand the file straight back rather than leaving it in a VM that is
# about to be reclaimed.
try:
    from google.colab import files
    files.download(str(dest))
except Exception as e:
    print("(not Colab, or download unavailable:", e, ")")